# 3.4.5 Feature Selection

Filter (SelectKBest), Wrapper (RFE), Embedded (Lasso), Permutation Importance.

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.feature_selection import SelectKBest,f_regression
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
h=fetch_california_housing(); X,y=h.data,h.target
for k in [2,4,6,8]:
    p=make_pipeline(StandardScaler(),SelectKBest(f_regression,k=k),Ridge(1.))
    s=cross_val_score(p,X,y,cv=5,scoring='neg_root_mean_squared_error')
    print(f'k={k}: RMSE={-s.mean():.4f}')

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split
Xt,Xv,yt,yv=train_test_split(X,y,test_size=.2,random_state=42)
sc=StandardScaler(); Xts=sc.fit_transform(Xt); Xvs=sc.transform(Xv)
rfe=RFE(Ridge(1.),n_features_to_select=4); rfe.fit(Xts,yt)
print('RFE selected:',list(np.array(h.feature_names)[rfe.support_]))

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error
for alpha in [0.01,0.05,0.1,0.5]:
    p=make_pipeline(StandardScaler(),Lasso(alpha,max_iter=5000)); p.fit(Xt,yt)
    nz=(p.named_steps['lasso'].coef_!=0).sum()
    print(f'Lasso(a={alpha}): {nz}/8 feats, RMSE={mean_squared_error(yv,p.predict(Xv))**.5:.4f}')

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
rf=RandomForestRegressor(50,random_state=42,n_jobs=-1); rf.fit(Xt,yt)
res=permutation_importance(rf,Xv,yv,n_repeats=10,random_state=42)
for i in res.importances_mean.argsort()[::-1]:
    print(f'{h.feature_names[i]:15s}: {res.importances_mean[i]:.4f}')